# Missense SNV Classification

Project notebook to build a missense SNV classification pipeline for PAH, CFTR, and hereditary cancer panel genes.


## 1. Set Up Paths, Seeds, and Target Genes

Define file paths, random seed, and the target gene list from the project plan.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\Users\Umut\Desktop\missense_classificaiton")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CLINVAR_SUMMARY = RAW_DIR / "variant_summary.txt.gz"
REF_FASTA = PROJECT_ROOT / "Homo_sapiens.GRCh38.dna.primary_assembly.fa"
PROTEIN_FASTA = PROJECT_ROOT / "idmapping_2026_03_16.fasta"
VEP_TSV = PROCESSED_DIR / "vep_results.tsv"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TARGET_GENES = [
    "PAH",
    "CFTR",
    "BRCA1",
    "BRCA2",
    "PALB2",
    "MLH1",
    "MSH2",
    "MSH6",
    "PMS2",
    "EPCAM",
    "TP53",
    "APC",
    "PTEN",
    "CDH1",
]


## 2. Load ClinVar Variant Summary

Read the local ClinVar variant_summary.txt.gz into a pandas DataFrame.


In [ ]:
if not CLINVAR_SUMMARY.exists():
    raise FileNotFoundError(f"Missing ClinVar summary at {CLINVAR_SUMMARY}")

clinvar_df = pd.read_csv(CLINVAR_SUMMARY, sep="\t", low_memory=False)
clinvar_df.head()


## 3. Filter ClinVar for Missense SNVs and Labels

Apply review status, variant type, consequence, gene panel, and label mapping filters.


In [ ]:
review_keep = {"reviewed by expert panel", "practice guideline"}
label_map = {
    "Benign": "Benign",
    "Likely benign": "Benign",
    "Pathogenic": "Pathogenic",
    "Likely pathogenic": "Pathogenic",
}

filtered = clinvar_df[
    clinvar_df["ReviewStatus"].isin(review_keep)
    & (clinvar_df["Type"] == "single nucleotide variant")
    & (clinvar_df["MolecularConsequence"].str.contains("missense", case=False, na=False))
    & (clinvar_df["GeneSymbol"].isin(TARGET_GENES))
].copy()

filtered["Label"] = filtered["ClinicalSignificance"].map(label_map)
filtered = filtered[filtered["Label"].notna()].copy()

filtered["Label"].value_counts(dropna=False)


## 4. Build Minimal VCF for VEP

Construct a minimal VCF with CHROM, POS, REF, ALT for offline VEP processing.


In [ ]:
vcf_cols = ["Chromosome", "PositionVCF", "ReferenceAllele", "AlternateAllele"]
missing_cols = [col for col in vcf_cols if col not in filtered.columns]
if missing_cols:
    raise KeyError(f"Missing columns for VCF: {missing_cols}")

vcf_df = filtered[vcf_cols].rename(
    columns={
        "Chromosome": "CHROM",
        "PositionVCF": "POS",
        "ReferenceAllele": "REF",
        "AlternateAllele": "ALT",
    }
)

vcf_path = PROCESSED_DIR / "clinvar_filtered.vcf"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

with open(vcf_path, "w", encoding="utf-8") as handle:
    handle.write("##fileformat=VCFv4.2\n")
    handle.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
    for row in vcf_df.itertuples(index=False):
        handle.write(f"{row.CHROM}\t{row.POS}\t.\t{row.REF}\t{row.ALT}\t.\t.\t.\n")

vcf_path


## 5. Load VEP TSV Output

Read the offline VEP output TSV and select required scores.


In [ ]:
if not VEP_TSV.exists():
    raise FileNotFoundError(f"Missing VEP TSV at {VEP_TSV}")

vep_df = pd.read_csv(VEP_TSV, sep="\t", low_memory=False, comment="#")

vep_keep = [
    "#Uploaded_variation",
    "Location",
    "Allele",
    "SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "CADD_phred",
    "REVEL_score",
    "MetaLR_score",
    "GERP++_RS",
    "phyloP100way_vertebrate",
    "phastCons100way_vertebrate",
    "gnomAD_AF",
]

vep_existing = [col for col in vep_keep if col in vep_df.columns]
vep_df = vep_df[vep_existing].copy()
vep_df.head()


## 6. Extract gnomAD AF from VEP

Parse global allele frequency from the VEP output and add missing-flag handling.


In [ ]:
if "gnomAD_AF" in vep_df.columns:
    vep_df["gnomAD_AF"] = pd.to_numeric(vep_df["gnomAD_AF"], errors="coerce")
    vep_df["gnomAD_AF_missing"] = vep_df["gnomAD_AF"].isna().astype(int)
    vep_df["gnomAD_AF"] = vep_df["gnomAD_AF"].fillna(0.0)
else:
    vep_df["gnomAD_AF"] = 0.0
    vep_df["gnomAD_AF_missing"] = 1

vep_df[["gnomAD_AF", "gnomAD_AF_missing"]].head()


## 7. Extract Nucleotide Context from GRCh38 FASTA

Use pysam to extract +/-5 nt windows and derive context features.


In [ ]:
import pysam

if not REF_FASTA.exists():
    raise FileNotFoundError(f"Missing reference FASTA at {REF_FASTA}")

fasta = pysam.FastaFile(str(REF_FASTA))

# Placeholder: implement window extraction using filtered variant positions.
# Example expected output columns: ["CHROM", "POS", "nt_window", "gc_content"]
nt_context = pd.DataFrame(columns=["CHROM", "POS", "nt_window", "gc_content"])
nt_context.head()


## 8. Extract Amino-Acid Context from Protein FASTA

Parse UniProt FASTA, map protein positions, and extract +/-5 AA windows.


In [ ]:
from Bio import SeqIO

if not PROTEIN_FASTA.exists():
    raise FileNotFoundError(f"Missing protein FASTA at {PROTEIN_FASTA}")

protein_records = {rec.id: str(rec.seq) for rec in SeqIO.parse(str(PROTEIN_FASTA), "fasta")}

# Placeholder: implement mapping from protein change to position, then window extraction.
# Example expected output columns: ["GeneSymbol", "ProteinPosition", "aa_window"]
aa_context = pd.DataFrame(columns=["GeneSymbol", "ProteinPosition", "aa_window"])
aa_context.head()


## 9. Compute Biochemical Substitution Features

Calculate delta scales and Grantham distance from ref/alt amino acids.


In [ ]:
# Placeholder: add lookup tables for biochemical properties and Grantham matrix.
# Expected input columns: ["RefAA", "AltAA"]
biochem_features = pd.DataFrame(columns=["RefAA", "AltAA", "delta_hydrophobicity", "delta_volume", "delta_charge", "delta_mw", "delta_polarity", "grantham_distance"])
biochem_features.head()


## 10. Assemble and Merge Feature Tables

Join ClinVar, VEP, nucleotide, amino-acid, and biochemical features into one table.


In [ ]:
# Placeholder merge logic. Replace join keys based on available columns.
features = filtered.copy()

# Example joins:
# features = features.merge(vep_df, left_on=["Chromosome", "PositionVCF"], right_on=["CHROM", "POS"], how="left")
# features = features.merge(nt_context, on=["CHROM", "POS"], how="left")
# features = features.merge(aa_context, on=["GeneSymbol", "ProteinPosition"], how="left")
# features = features.merge(biochem_features, on=["RefAA", "AltAA"], how="left")

features.head()


## 11. Drop Genomic Address Columns

Remove CHROM, POS, and other genomic address columns after merging.


In [ ]:
genomic_cols = [
    "Chromosome",
    "PositionVCF",
    "CHROM",
    "POS",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
]

features_final = features.drop(columns=[col for col in genomic_cols if col in features.columns])
features_final.head()
